In [0]:
%sql
CREATE TABLE IF NOT EXISTS stackoverflow_questions (
    question_id TEXT PRIMARY KEY,
    title TEXT NOT NULL,
    body TEXT,
    tags TEXT,
    link TEXT,
    score INT,
    view_count INT,
    answer_count INT,
    creation_date TIMESTAMPTZ,
    last_activity_date TIMESTAMPTZ,
    owner_display_name TEXT,
    owner_reputation INT,
    is_answered BOOLEAN,
    payload JSONB,
    synced_at TIMESTAMPTZ DEFAULT CURRENT_TIMESTAMP
);

CREATE INDEX IF NOT EXISTS idx_so_tags ON stackoverflow_questions USING GIN(tags);
CREATE INDEX IF NOT EXISTS idx_so_score ON stackoverflow_questions(score DESC);

-- stackoverflow_embeddings table (requires pgvector extension)
CREATE EXTENSION IF NOT EXISTS vector;

CREATE TABLE IF NOT EXISTS stackoverflow_embeddings (
    id SERIAL PRIMARY KEY,
    question_id TEXT NOT NULL REFERENCES stackoverflow_questions(question_id) ON DELETE CASCADE,
    embedding vector(384),  -- adjust dimension based on your model
    text_content TEXT NOT NULL,
    created_at TIMESTAMPTZ DEFAULT CURRENT_TIMESTAMP,
    UNIQUE(question_id)
);

CREATE INDEX IF NOT EXISTS idx_so_embedding_vector 
    ON stackoverflow_embeddings USING ivfflat (embedding vector_cosine_ops);